# 🎬 LongCat Studio SE — Talking Avatar

**Ladda upp ett foto + skriv text → Få en talande video**

Baserat på [LongCat-Video-Avatar 1.5](https://github.com/meituan-longcat/LongCat-Video) av Meituan (MIT-licens).

---

## ⚠️ GPU-krav
| GPU | VRAM | Status |
|-----|------|--------|
| **A100 (40GB)** | 40 GB | ✅ Rekommenderas |
| **V100 (32GB)** | 32 GB | ✅ Fungerar |
| T4 (15GB) | 15 GB | ❌ För liten |

> **Aktivera A100:** Runtime → Change runtime type → A100 *(kräver Colab Pro, ~10 USD/mån)*

---

## ⏱️ Tidsuppskattning
- **Första sessionen:** ~20 min setup (installation + nedladdning av ~25 GB modell)
- **Efterföljande sessioner:** ~5 min setup (modeller hämtas från Google Drive)
- **Per video:** ~8–18 min (inkl. modelladdning ~3–8 min + generering ~5–10 min)

---

## 🔒 GDPR / Dataskydd
- Bilder och ljud bearbetas lokalt i din Colab-VM (rensas när sessionen avslutas)
- Modeller lagras i din Google Drive (ditt eget konto)
- **OBS:** TTS-text skickas till Microsoft Edge TTS API (speech.platform.bing.com) för röstsyntes. Inkludera **inte** känsliga personuppgifter i TTS-texten.

---

## Workflow
1. Kör **Steg 1–7** en gång (tar ~20 min första gången, sedan < 5 min via Drive-cache)
2. Kör **Steg 8** för att starta Gradio-studion
3. Ladda upp bild + skriv svensk text → klicka Generera → ladda ner video

In [ ]:
# ── Steg 1: GPU-check ──────────────────────────────────────────────────
import torch, subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:800])

if not torch.cuda.is_available():
    raise RuntimeError("❌ Ingen GPU hittad! Aktivera GPU: Runtime → Change runtime type")

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"\nGPU: {gpu_name}")
print(f"VRAM: {gpu_mem_gb:.1f} GB")

if gpu_mem_gb < 30:
    print("\n❌ VARNING: Behöver minst 32 GB VRAM (A100/V100).")
    print("   Gå till: Runtime → Change runtime type → A100")
    raise RuntimeError("Otillräcklig VRAM")
else:
    print("\n✅ GPU OK — tillräcklig VRAM för LongCat Avatar 1.5")

In [ ]:
# ── Steg 2: Systempaket ────────────────────────────────────────────────
import subprocess

def run(cmd_list, desc=""):
    print(f"▶ {desc}")
    r = subprocess.run(cmd_list, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  STDERR: {r.stderr[-300:]}")
    else:
        print("  ✅ OK")
    return r.returncode

run(["apt-get", "install", "-qq", "ffmpeg"], "ffmpeg")
print("✅ Systempaket klara")

In [ ]:
# ── Steg 3: Python-paket ────────────────────────────────────────────────
import subprocess, sys

def pip_install(pkg, desc=""):
    print(f"▶ {desc or pkg}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkg.split(),
                       capture_output=True, text=True)
    print("  ✅ OK" if r.returncode == 0 else f"  ⚠️ {r.stderr[-200:]}")
    return r.returncode

pkgs_base = [
    ("ninja psutil packaging",       "Byggverktyg"),
    ("huggingface_hub[cli]",          "HuggingFace CLI"),
    ("edge-tts nest_asyncio",         "Svensk TTS (Microsoft Edge — gratis)"),
    ("soundfile librosa",             "Audiobearbetning"),
    ("gradio>=4.0",                   "Gradio UI"),
    ("audio-separator",               "Röstseparation"),
]

for pkg, desc in pkgs_base:
    pip_install(pkg, desc)

# flash-attn — valfritt, faller tillbaka på sdpa om det misslyckas
print("▶ flash-attn (kan ta 2-3 min att kompilera, hoppar över om det misslyckas)")
rc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "flash_attn==2.7.4.post1", "--no-build-isolation"],
    capture_output=True, text=True
).returncode

FLASH_ATTN_OK = (rc == 0)
print(f"  flash-attn: {'✅' if FLASH_ATTN_OK else '⚠️ ej tillgänglig — patchar config till sdpa'}")

print("\n✅ Python-paket klara")

In [ ]:
# ── Steg 4: Klona LongCat ──────────────────────────────────────────────
import os, subprocess, sys

LONGCAT_DIR = "/content/LongCat-Video"

if not os.path.exists(LONGCAT_DIR):
    print("▶ Klonar LongCat-Video...")
    subprocess.run(
        ["git", "clone", "-q", "--single-branch", "--branch", "main",
         "https://github.com/meituan-longcat/LongCat-Video", LONGCAT_DIR],
        check=True
    )
    print("  ✅ Klonad")
else:
    print("  ✅ LongCat-Video finns redan")

os.chdir(LONGCAT_DIR)

print("▶ Installerar LongCat requirements...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],      check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements_avatar.txt"], check=True)
print("✅ LongCat installerat — katalog:", os.getcwd())

In [ ]:
# ── Steg 5: Montera Google Drive (modell-cache) ────────────────────────
# Modellerna (~25 GB) laddas ner EN gång och sparas i Drive.
# Nästa session: startar direkt utan nedladdning.

from google.colab import drive
import os

LONGCAT_DIR  = "/content/LongCat-Video"
drive.mount('/content/drive')

DRIVE_MODELS = "/content/drive/MyDrive/LongCat-Models"
os.makedirs(DRIVE_MODELS, exist_ok=True)

WEIGHTS_DIR = f"{LONGCAT_DIR}/weights"

# Skapa symlänk LongCat/weights → Drive (om den inte redan finns)
# Kollar islink() för att hantera dangling symlinks korrekt
if not os.path.islink(WEIGHTS_DIR) and not os.path.exists(WEIGHTS_DIR):
    os.symlink(DRIVE_MODELS, WEIGHTS_DIR)
    print(f"✅ weights/ → {DRIVE_MODELS}")
else:
    print(f"✅ weights/ finns redan (islink={os.path.islink(WEIGHTS_DIR)})")

print(f"Modeller sparas i: {DRIVE_MODELS}")

In [ ]:
# ── Steg 6: Ladda ner modeller ─────────────────────────────────────────
# Första gången: ~25 GB, tar 10-20 min beroende på nätverkshastighet.
# Efterföljande sessioner: direkt från Drive (< 1 min).

import os, subprocess, json, sys

# Säkert standardvärde om Steg 3 inte körts i samma session
FLASH_ATTN_OK = globals().get('FLASH_ATTN_OK', False)

DRIVE_MODELS = "/content/drive/MyDrive/LongCat-Models"
BASE_MODEL   = f"{DRIVE_MODELS}/LongCat-Video"
AVATAR_MODEL = f"{DRIVE_MODELS}/LongCat-Video-Avatar-1.5"

def download_if_missing(repo_id, local_dir):
    marker = os.path.join(local_dir, ".download_complete")
    if os.path.exists(marker):
        print(f"  ✅ {os.path.basename(local_dir)} — hittad i Drive-cache")
        return
    print(f"  ⬇️  Laddar ner {repo_id} → {local_dir}")
    print("     (kan ta 5-15 min — avbryt inte Colab-sessionen)")
    subprocess.run(
        ["huggingface-cli", "download", repo_id, "--local-dir", local_dir],
        check=True
    )
    # Marker skrivs EFTER slutförd nedladdning
    open(marker, 'w').close()
    print(f"  ✅ {os.path.basename(local_dir)} klar")

print("=== Modell-nedladdning ===")
download_if_missing("meituan-longcat/LongCat-Video",            BASE_MODEL)
download_if_missing("meituan-longcat/LongCat-Video-Avatar-1.5", AVATAR_MODEL)

# Patcha config om flash-attn saknas
for config_path in [
    f"{BASE_MODEL}/dit/config.json",
    f"{AVATAR_MODEL}/base_model/config.json",
    f"{AVATAR_MODEL}/base_model_int8/config.json",
]:
    if os.path.exists(config_path):
        with open(config_path) as f:
            cfg = json.load(f)
        if not FLASH_ATTN_OK and cfg.get('attn_implementation') == 'flash_attention_2':
            cfg['attn_implementation'] = 'sdpa'
            with open(config_path, 'w') as f:
                json.dump(cfg, f, indent=2)
            print(f"  Patched {os.path.basename(config_path)}: flash_attention_2 → sdpa")

print("\n✅ Alla modeller redo!")

In [ ]:
# ── Steg 7: Verifiera installation ─────────────────────────────────────
import os, sys, torch

LONGCAT_DIR  = "/content/LongCat-Video"
DRIVE_MODELS = "/content/drive/MyDrive/LongCat-Models"

checks = [
    (f"{LONGCAT_DIR}/longcat_video",                                          "LongCat Python-modul"),
    (f"{LONGCAT_DIR}/run_demo_avatar_single_audio_to_video.py",               "Inference-skript"),
    (f"{DRIVE_MODELS}/LongCat-Video",                                         "Basmodell (Drive)"),
    (f"{DRIVE_MODELS}/LongCat-Video-Avatar-1.5",                              "Avatar 1.5 modell (Drive)"),
    ("/usr/bin/ffmpeg",                                                         "ffmpeg"),
]

all_ok = True
for path, name in checks:
    ok = os.path.exists(path)
    print(f"{'✅' if ok else '❌'} {name}")
    if not ok:
        all_ok = False
        print(f"   Saknas: {path}")

# Testa imports
try:
    sys.path.insert(0, LONGCAT_DIR)
    from longcat_video.pipeline_longcat_video_avatar import LongCatVideoAvatarPipeline
    print("✅ LongCatVideoAvatarPipeline importerad")
except Exception as e:
    print(f"❌ Import-fel: {e}")
    all_ok = False

try:
    import edge_tts
    print("✅ edge-tts importerat")
except ImportError:
    print("❌ edge-tts saknas — kör Steg 3 igen")
    all_ok = False

if all_ok:
    print("\n🎉 INSTALLATION KOMPLETT — Kör Steg 8 för att starta studion!")
    print("   Obs: Varje videogenerering laddar om modellen (~3–8 min modelladdning + ~5–10 min generering)")
else:
    print("\n⚠️ Något saknas — kontrollera felmeddelandena ovan")

In [ ]:
# ── Steg 8: LongCat Studio — Gradio UI ────────────────────────────────
# OBS: torchrun startar en ny process per videogenerering.
# Modellen (~25 GB) laddas om varje gång: ~3–8 min modelladdning + ~5–10 min generering.

import os, sys, json, asyncio, tempfile, subprocess, shutil
import nest_asyncio
import gradio as gr
import edge_tts

nest_asyncio.apply()

LONGCAT_DIR     = "/content/LongCat-Video"
CHECKPOINT_DIR  = "/content/drive/MyDrive/LongCat-Models/LongCat-Video-Avatar-1.5"

sys.path.insert(0, LONGCAT_DIR)
os.chdir(LONGCAT_DIR)

TTS_MAX_CHARS = 2000  # Max teckengräns för TTS (Microsoft API-begränsning)

# ── Svenska Edge-TTS röster ────────────────────────────────────────────
# Röster via Microsoft Edge TTS (gratis, ingen API-nyckel)
# OBS: Text skickas till speech.platform.bing.com — inkludera inte känsliga data
SWEDISH_VOICES = {
    "Sofie (kvinna, rikssvenska)": "sv-SE-SofieNeural",
    "Mattias (man, rikssvenska)": "sv-SE-MattiasNeural",
    "Noora (kvinna, finlandssvenska)": "sv-FI-NooraNeural",
}

STANDARD_PROMPT = (
    "A professional person is speaking clearly and naturally, "
    "making eye contact, natural expressions, high quality, realistic."
)

# ── TTS: text → WAV 16kHz ─────────────────────────────────────────────
async def _tts_async(text: str, voice: str, out_wav: str):
    communicate = edge_tts.Communicate(text, voice)
    mp3_path = out_wav.replace('.wav', '.mp3')
    await communicate.save(mp3_path)
    # Konvertera MP3 → WAV 16kHz mono (LongCat-krav)
    subprocess.run(
        ["ffmpeg", "-y", "-i", mp3_path, "-ar", "16000", "-ac", "1", out_wav],
        check=True, capture_output=True
    )
    os.remove(mp3_path)

def text_to_speech(text: str, voice_label: str) -> str:
    voice    = SWEDISH_VOICES.get(voice_label, "sv-SE-SofieNeural")
    out_path = tempfile.mktemp(suffix=".wav")
    asyncio.get_event_loop().run_until_complete(_tts_async(text, voice, out_path))
    return out_path

# ── Generera video via torchrun ────────────────────────────────────────
def generate_video(
    image_path,
    text,
    voice_label,
    custom_audio,
    prompt,
    resolution,
    num_segments,
    progress=gr.Progress()
):
    if image_path is None:
        return None, "❌ Ladda upp ett porträttfoto."
    if not str(text).strip() and custom_audio is None:
        return None, "❌ Ange text ELLER ladda upp ett ljud-klipp."
    if len(str(text)) > TTS_MAX_CHARS:
        return None, f"❌ Text för lång ({len(text)} tecken). Max {TTS_MAX_CHARS} tecken."

    tmp_dir = tempfile.mkdtemp()
    out_dir = os.path.join(tmp_dir, "output")
    os.makedirs(out_dir, exist_ok=True)

    try:
        # 1. Generera eller konvertera ljud
        progress(0.05, "Förbereder ljud...")
        audio_path = os.path.join(tmp_dir, "audio.wav")

        if custom_audio is not None:
            # Konvertera uppladdad fil till 16kHz WAV
            subprocess.run(
                ["ffmpeg", "-y", "-i", custom_audio, "-ar", "16000", "-ac", "1", audio_path],
                check=True, capture_output=True
            )
        else:
            progress(0.1, "Genererar svenska TTS-röst (Microsoft Edge TTS)...")
            tts_wav = text_to_speech(str(text), voice_label)
            shutil.move(tts_wav, audio_path)

        # 2. Kopiera bild till tmpdir
        img_ext  = os.path.splitext(image_path)[1] or ".jpg"
        img_path = os.path.join(tmp_dir, f"portrait{img_ext}")
        shutil.copy(image_path, img_path)

        # 3. Skapa input JSON
        # Schema: https://github.com/meituan-longcat/LongCat-Video (run_demo_avatar_single)
        input_json_path = os.path.join(tmp_dir, "input.json")
        input_data = {
            "prompt": (prompt or STANDARD_PROMPT).strip(),
            "cond_audio": {"person1": audio_path},
            "cond_image": img_path
        }
        with open(input_json_path, 'w') as f:
            json.dump(input_data, f)

        # 4. Kör LongCat-inference via torchrun (single GPU)
        # --stage_1=ai2v = Audio Image to Video (avatar mode)
        # --model_type=avatar-v1.5 = Whisper-large-v3 audio encoder
        # --use_int8 + --use_distill = INT8-kvantisering + 8-stegs distillation (snabb)
        progress(0.15, "Laddar LongCat-modell (~3–8 min första gången)...")

        cmd = [
            "torchrun", "--nproc_per_node=1",
            "run_demo_avatar_single_audio_to_video.py",
            "--context_parallel_size", "1",
            "--checkpoint_dir", CHECKPOINT_DIR,
            "--stage_1", "ai2v",
            "--input_json", input_json_path,
            "--num_segments", str(int(num_segments)),
            "--resolution", resolution,
            "--use_distill",
            "--model_type", "avatar-v1.5",
            "--use_int8",
            "--output_dir", out_dir,
        ]

        print(f"\nKör: {' '.join(cmd)}\n")
        result = subprocess.run(cmd, cwd=LONGCAT_DIR)

        if result.returncode != 0:
            return None, f"❌ Inference misslyckades (exit {result.returncode}). Se logg ovan."

        # 5. Hitta output-video (väljer den med störst filstorlek)
        progress(0.95, "Söker output-video...")
        videos = [
            os.path.join(out_dir, f)
            for f in os.listdir(out_dir)
            if f.endswith('.mp4') and os.path.getsize(os.path.join(out_dir, f)) > 10_000
        ]

        if not videos:
            return None, "❌ Ingen video genererades — kontrollera loggarna ovan."

        video_path = max(videos, key=os.path.getsize)
        progress(1.0, "Klar!")
        size_mb = os.path.getsize(video_path) / 1024**2
        return video_path, f"✅ Video klar! {os.path.basename(video_path)} ({size_mb:.1f} MB)"

    except Exception as e:
        return None, f"❌ Fel: {str(e)}"
    finally:
        # Städa upp tempfiler (audio, json — behåll video tills Gradio är klar med den)
        for f in ["audio.wav", "input.json", "portrait.jpg", "portrait.png"]:
            p = os.path.join(tmp_dir, f)
            if os.path.exists(p):
                os.remove(p)


# ── Gradio UI ──────────────────────────────────────────────────────────
with gr.Blocks(
    title="LongCat Studio SE",
    theme=gr.themes.Soft(primary_hue="blue"),
) as demo:

    gr.Markdown(
        """# 🎬 LongCat Studio SE
        ### Ladda upp ett porträttfoto · Skriv vad personen ska säga · Få en talande video
        *Bygger på [LongCat-Video-Avatar 1.5](https://github.com/meituan-longcat/LongCat-Video) (MIT) · """
        """TTS via [Microsoft Edge TTS](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/) (gratis)*

        > ⏱️ **Tidsuppskattning:** ~3–8 min modelladdning + ~5–10 min generering per video (A100)
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Input")

            image_input = gr.Image(
                label="Porträttfoto (JPG/PNG) — neutral bakgrund, tydligt ansikte",
                type="filepath",
                height=280,
            )

            with gr.Tab("✍️ Text → Röst (TTS)"):
                text_input = gr.Textbox(
                    label=f"Text på svenska (max {TTS_MAX_CHARS} tecken)",
                    placeholder="Hej! Jag heter Anna och vill berätta om hur AI kan hjälpa ditt företag...",
                    lines=4,
                )
                voice_select = gr.Dropdown(
                    choices=list(SWEDISH_VOICES.keys()),
                    value=list(SWEDISH_VOICES.keys())[0],
                    label="Röst (Microsoft Edge TTS — text skickas till Microsoft)",
                )

            with gr.Tab("🎤 Eget ljud"):
                audio_input = gr.Audio(
                    label="Ladda upp ljud (WAV/MP3) — ersätter TTS",
                    type="filepath",
                )

            with gr.Accordion("⚙️ Avancerade inställningar", open=False):
                prompt_input = gr.Textbox(
                    label="Video-prompt (valfri — beskriv scen, klädsel, miljö)",
                    value=STANDARD_PROMPT,
                    lines=3,
                )
                resolution_select = gr.Radio(
                    choices=["480p", "720p"],
                    value="480p",
                    label="Upplösning (720p kräver mer VRAM, tar längre tid)",
                )
                segments_input = gr.Slider(
                    minimum=1, maximum=5, step=1, value=1,
                    label="Antal segment (1 segment ≈ 3,7 sek video)",
                )

            generate_btn = gr.Button(
                "🎬  Generera video",
                variant="primary",
                size="lg"
            )

        with gr.Column(scale=1):
            gr.Markdown("### Output")

            video_output = gr.Video(
                label="Genererad video",
                height=380,
            )
            status_output = gr.Textbox(
                label="Status",
                interactive=False,
                lines=2,
            )

            gr.Markdown(
                """#### Tips för bästa resultat
                - **Foto:** Neutral bakgrund, bra frontalbelysning, synlig hals
                - **Text:** Naturliga meningar, rimlig längd (30–200 ord)
                - **Prompt:** Beskriv scen + klädsel → naturligare video
                - **Segment:** 1 segment = ~3,7 sek · 3 segment = ~11 sek
                - **480p** räcker för sociala medier och LinkedIn
                """
            )

    gr.Markdown("---\n### Exempel-prompts (byt ut bilden mot ditt foto)")
    gr.Examples(
        examples=[
            [
                None,
                "Hej, jag heter Anna och vill berätta om hur AI kan revolutionera din verksamhet. Kontakta oss idag för en kostnadsfri demo!",
                list(SWEDISH_VOICES.keys())[0],
                None,
                "A professional woman is speaking confidently, wearing a white blouse, bright modern office, natural expressions.",
                "480p",
                1,
            ],
            [
                None,
                "Välkommen till vår butik! Vi har precis fått in nya produkter som jag tror du kommer att älska. Kom in och titta!",
                list(SWEDISH_VOICES.keys())[1],
                None,
                "A friendly man is speaking warmly, wearing casual clothing, bright store interior background, smiling.",
                "480p",
                1,
            ],
        ],
        inputs=[
            image_input, text_input, voice_select, audio_input,
            prompt_input, resolution_select, segments_input
        ],
        label="Klicka för att fylla i exempeldata",
    )

    generate_btn.click(
        fn=generate_video,
        inputs=[
            image_input, text_input, voice_select, audio_input,
            prompt_input, resolution_select, segments_input
        ],
        outputs=[video_output, status_output],
        show_progress="full",
    )

print("🚀 Startar LongCat Studio SE...")
demo.launch(share=True, debug=False)